# Build the CauseTrace graph, embeddings and runtime files

Run from the repository root. Needs the original dataset's `analysis_data/core/graph_aggregated/*.parquet` shards (not included in this repository; see the README for attribution and terms).

Produces `data/macro_claims.parquet`, `indexes/kg_nodes.json` and `indexes/kg_bge_m3.faiss`. Embedding all nodes takes a long time on a laptop. Outputs of the original run were cleared for publication.


In [ ]:
import pandas as pd
import networkx as nx
from pathlib import Path
import json
from ollama import chat
import numpy as np
import time
import faiss
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# Set this to your existing CausalClaimsInEconomics folder.
DATA_DIR = Path("analysis_data/core/graph_aggregated")

files = sorted(DATA_DIR.glob("*.parquet"))

print("Parquet files found:", len(files))
for f in files:
    print(f)
    
if not files:
    raise FileNotFoundError(
        "Parquet files not found. Check DATA_DIR."
    )

df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print("Total claims:", len(df))
print("Columns:", df.columns.tolist())


In [ ]:
# Keep only macroeconomic claims.
macro = df.loc[df["is_macroeconomics"] == True].copy()

# Normalize text consistently with our previous notebook.
for col in ["cause", "effect"]:
    macro[col] = (
        macro[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

macro = macro.dropna(subset=["cause", "effect"])
macro = macro[
    (macro["cause"] != "") &
    (macro["effect"] != "")
]

# Build the directed knowledge graph.
G = nx.from_pandas_edgelist(
    macro,
    source="cause",
    target="effect",
    create_using=nx.DiGraph()
)

print("Macro claims:", len(macro))
print("KG nodes:", G.number_of_nodes())
print("KG edges:", G.number_of_edges())


In [ ]:
print("Graph type:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\nEvidence columns:")
print(macro.columns.tolist())

print("\nSource / Target present:")
print("economic growth" in G)
print("stock prices" in G)
print("equity returns" in G)


In [ ]:
SYSTEM_PROMPT = """
You are a precise economic query parser.
Your ONLY task is to extract information explicitly
stated in the user's question.

Return a JSON object with exactly three fields:
- source: string or null
- target: string or null
- query_type: "cause_to_effect", "open_ended", or "other"

Rules:
1. Identify what causes or changes something (source).
2. Identify what is affected (target).
3. In "How does X affect Y?", X is source and Y is target.
4. In "How might X influence Y?", X is source and Y is target.
5. In "What is the impact of X on Y?", X is source and Y is target.
6. If no specific outcome Y is named, target must be null.
7. Generic terms like "economic consequences", "effects",
   "implications", and "impacts" are NOT specific targets.
8. If target is null and source exists, use "open_ended".
9. If source and specific target exist, use "cause_to_effect".
10. Do not infer or invent missing economic concepts.
11. Preserve modifiers such as positive, higher, lower,
    unexpected, and surprise.
12. Never omit an explicitly mentioned target.
13. Return JSON only.

Examples:

Question: How does inflation affect unemployment?
Answer: {"source":"inflation","target":"unemployment",
"query_type":"cause_to_effect"}

Question: What are the economic consequences of higher oil prices?
Answer: {"source":"higher oil prices","target":null,
"query_type":"open_ended"}

Question: How does a positive GDP growth surprise affect equities?
Answer: {"source":"positive GDP growth surprise",
"target":"equities","query_type":"cause_to_effect"}

Question: What happens when the Fed unexpectedly raises rates?
Answer: {"source":"unexpected Fed rate increase",
"target":null,"query_type":"open_ended"}
"""

def parse_query(query):
    response = chat(
        model="qwen3:4b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ],
        format="json",
        options={"temperature": 0}
    )

    result = json.loads(response.message.content)

    # Ensure query_type is consistent with the extracted fields.
    if result.get("source") and result.get("target"):
        result["query_type"] = "cause_to_effect"
    elif result.get("source") and not result.get("target"):
        result["query_type"] = "open_ended"

    return result


In [ ]:
queries = [
    "What are the economic consequences of higher oil prices?",
    "How does a positive GDP growth surprise affect equities?",
    "What happens when the Fed unexpectedly raises interest rates?"
]

for query in queries:
    print("\nQuestion:", query)
    print("Parsed:", parse_query(query))


In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=False
)

print("BGE-M3 loaded!")


In [ ]:
# Sample nodes from the existing KG
test_nodes = sorted(str(n) for n in G.nodes())[:1000]

start = time.time()

output = model.encode(
    test_nodes,
    batch_size=16,
    max_length=64
)

embeddings = np.asarray(
    output["dense_vecs"],
    dtype="float32"
)

print("Embedding shape:", embeddings.shape)
print("Elapsed seconds:", round(time.time() - start, 2))


In [ ]:
INDEX_DIR = Path("indexes")
INDEX_DIR.mkdir(exist_ok=True)

kg_nodes = sorted(str(n) for n in G.nodes())

# Use normalized vectors for cosine similarity.
index = faiss.IndexFlatIP(1024)

batch_size = 32

for start in tqdm(range(0, len(kg_nodes), batch_size)):
    batch = kg_nodes[start:start + batch_size]

    output = model.encode(
        batch,
        batch_size=16,
        max_length=64
    )

    vectors = np.asarray(
        output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(vectors)
    index.add(vectors)

# Save index and exact node ordering.
faiss.write_index(index, str(INDEX_DIR / "kg_bge_m3.faiss"))

with open(INDEX_DIR / "kg_nodes.json", "w", encoding="utf-8") as f:
    json.dump(kg_nodes, f, ensure_ascii=False)

print("Indexed nodes:", index.ntotal)
print("Saved to:", INDEX_DIR.resolve())


In [ ]:
def search_nodes(query, top_k=10):
    output = model.encode_single_device(
        [query],
        device="cpu",
        batch_size=1,
        max_length=64,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.array(
        output["dense_vecs"],
        dtype="float32",
        copy=True
    )
    faiss.normalize_L2(query_vector)

    scores, ids = index.search(
        query_vector,
        min(top_k, index.ntotal)
    )

    return [
        {
            "node": kg_nodes[int(i)],
            "similarity": float(score)
        }
        for score, i in zip(scores[0], ids[0])
        if i >= 0
    ]


for query in ["economic growth", "inflation", "stock prices"]:
    print(f"\nQUERY: {query}")

    for result in search_nodes(query):
        print(f"{result['similarity']:.3f}  {result['node']}")

In [ ]:
source = "economic growth"
target = "stock prices"

assert G.is_directed(), "A directed graph is required to inspect causal direction."
assert source in G, f"Missing node: {source}"
assert target in G, f"Missing node: {target}"

candidate_paths = []

# 1. Direct connection: source → target
if G.has_edge(source, target):
    candidate_paths.append([source, target])

# 2. One intermediate node: source → middle → target
middle_nodes = (
    set(G.successors(source))
    & set(G.predecessors(target))
) - {source, target}

for middle in sorted(middle_nodes, key=str):
    candidate_paths.append([source, middle, target])

print("Total short paths:", len(candidate_paths))

# Display up to 20 paths.
for i, path in enumerate(candidate_paths[:20], start=1):
    print(f"{i}. " + " → ".join(map(str, path)))

# Inspect available metadata fields for the first path.
if candidate_paths:
    print("\nMetadata fields on the first path:")

    path = candidate_paths[0]
    for u, v in zip(path, path[1:]):
        edge_data = G.get_edge_data(u, v)

        records = (
            list(edge_data.values())
            if G.is_multigraph()
            else [edge_data]
        )

        fields = sorted({
            str(key)
            for record in records
            for key in record
        })

        print(f"{u} → {v}")
        print("  Edge records:", len(records))
        print("  Available fields:", fields)

In [ ]:
print("Available columns:")
print(macro.columns.tolist())

edges_to_check = [
    ("economic growth", "government spending"),
    ("government spending", "stock prices"),
]

evidence_by_edge = {}

for cause, effect in edges_to_check:
    matches = macro.loc[
        macro["cause"].eq(cause)
        & macro["effect"].eq(effect)
    ].copy()

    evidence_by_edge[(cause, effect)] = matches

    print(f"\n{cause} → {effect}")
    print("Matching rows:", len(matches))

    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", 200
    ):
        display(matches.head(5))

In [ ]:
columns_to_show = [
    "paper_id",
    "title",
    "year",
    "cause",
    "effect",
    "is_method_causal_inference",
    "causal_inference_method",
    "sources_of_exogenous_variation",
]

for (cause, effect), matches in evidence_by_edge.items():
    print(f"\n{cause} → {effect}")
    print("Matching rows:", len(matches))
    print("Unique paper IDs:", matches["paper_id"].nunique())

    for record_number, (_, row) in enumerate(
        matches[columns_to_show].iterrows(),
        start=1
    ):
        print(f"\nRecord {record_number}")
        for column in columns_to_show:
            print(f"{column}: {row[column]}")

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

claims_path = data_dir / "macro_claims.parquet"
macro.to_parquet(claims_path, index=False)

required_files = [
    claims_path,
    Path("indexes/kg_bge_m3.faiss"),
    Path("indexes/kg_nodes.json"),
]

print("Project directory:", Path.cwd())
print("Saved claim rows:", len(macro))

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    print(f"{status}: {path.resolve()}")